# FinnGen Evidence Investigation Workbench

Welcome to the **FinnGen Evidence Investigation** workspace — Parthenon's platform for translational research combining clinical phenotyping, observational analytics, and genomic evidence.

This notebook provides direct access to:

1. **Connect** — database, Parthenon API, and FinnGen services
2. **Investigations** — create and manage evidence investigations via the API
3. **Phenotype domain** — OMOP concept search, cohort building, concept sets
4. **Clinical domain** — cohort operations, CO2 analysis, HADES analytics
5. **Genomic domain** — Open Targets, GWAS Catalog, colocalization queries
6. **Evidence synthesis** — cross-domain analysis and dossier assembly
7. **Publication-ready charts** — dark clinical theme matching Parthenon UI

---

**The Investigation Model:** Each study is a persistent *Evidence Investigation* — a named workspace where you work across four evidence domains (Phenotype, Clinical, Genomic, Synthesis), accumulating findings into an exportable Evidence Dossier.

## 1. Environment & Connection Setup

Run this cell first — it establishes your database connection, API client, and FinnGen service helpers.

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

# ── Paths ──
NOTEBOOK_DIR = Path(os.environ.get('PARTHENON_NOTEBOOK_DIR', '/home/jovyan/notebooks'))
SHARED_DIR = Path('/home/jovyan/shared')
USER_EMAIL = os.environ.get('PARTHENON_USER_EMAIL', '')

# ── Database ──
DB_USER = os.environ.get('PARTHENON_DB_USER', 'parthenon')
DB_PASS = os.environ.get('PARTHENON_DB_PASSWORD', '')
DB_HOST = os.environ.get('PARTHENON_DB_HOST', 'postgres')
DB_PORT = os.environ.get('PARTHENON_DB_PORT', '5432')
DB_NAME = os.environ.get('PARTHENON_DB_NAME', 'parthenon')

DATABASE_URL = f'postgresql+psycopg://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DATABASE_URL)

# ── API client ──
API_BASE = os.environ.get('PARTHENON_API_BASE_URL', 'http://nginx/api/v1')
API_TOKEN = os.environ.get('PARTHENON_API_TOKEN', '')

def api_request(method: str, path: str, params: dict | None = None, json: dict | None = None) -> dict:
    """Make an authenticated request to the Parthenon API."""
    headers = {'Accept': 'application/json', 'Content-Type': 'application/json'}
    if API_TOKEN:
        headers['Authorization'] = f'Bearer {API_TOKEN}'
    url = urljoin(API_BASE.rstrip('/') + '/', path.lstrip('/'))
    resp = requests.request(method, url, params=params, json=json, headers=headers, timeout=60)
    resp.raise_for_status()
    return resp.json()

def api_get(path: str, params: dict | None = None) -> dict:
    return api_request('GET', path, params=params)

def api_post(path: str, json: dict | None = None) -> dict:
    return api_request('POST', path, json=json)

def api_patch(path: str, json: dict | None = None) -> dict:
    return api_request('PATCH', path, json=json)

def query(sql: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

# ── Verify ──
print(f'\u2713 User: {USER_EMAIL}')
print(f'\u2713 DB role: {DB_USER}')

with engine.connect() as conn:
    result = conn.execute(text('SELECT current_user, current_database()'))
    row = result.fetchone()
    print(f'\u2713 Connected to {row[1]} as {row[0]}')

# Check FinnGen services
try:
    services = api_get('/study-agent/services')
    svc_list = services.get('data', {}).get('services', [])
    fg_services = [s['name'] for s in svc_list if s.get('name', '').startswith('finngen_')]
    print(f'\u2713 FinnGen services: {fg_services}')
except Exception as e:
    print(f'\u26A0 FinnGen services unavailable: {e}')

## 2. Evidence Investigations

Create, list, and manage persistent evidence investigations — the unit of work in the FinnGen workbench.

In [ ]:
def list_investigations(status: str | None = None) -> pd.DataFrame:
    """List all evidence investigations."""
    params = {'status': status} if status else None
    data = api_get('/investigations', params=params)
    items = data.get('data', data) if isinstance(data, dict) else data
    if isinstance(items, list):
        return pd.DataFrame(items)
    return pd.DataFrame(items.get('data', []))

def create_investigation(title: str, question: str = '') -> dict:
    """Create a new evidence investigation."""
    payload = {'title': title}
    if question:
        payload['research_question'] = question
    return api_post('/investigations', json=payload)

def get_investigation(inv_id: int) -> dict:
    """Fetch a single investigation with full state."""
    return api_get(f'/investigations/{inv_id}')

def get_pins(inv_id: int) -> pd.DataFrame:
    """Get all evidence pins for an investigation."""
    data = api_get(f'/investigations/{inv_id}/pins')
    pins = data.get('data', data)
    return pd.DataFrame(pins) if pins else pd.DataFrame()

print('\u2713 Investigation helpers ready')

In [ ]:
# List existing investigations
investigations = list_investigations()
if investigations.empty:
    print('No investigations yet — create one below!')
else:
    display(investigations[['id', 'title', 'status', 'research_question', 'created_at']].head(10))

In [ ]:
# Create a new investigation (uncomment to run)
# inv = create_investigation(
#     title='SGLT2 Inhibitors and CKD Progression',
#     question='Does SGLT2 inhibition reduce CKD progression in patients with Type 2 Diabetes?'
# )
# print(f'Created investigation #{inv["data"]["id"]}: {inv["data"]["title"]}')

## 3. Phenotype Domain — OMOP Vocabulary & Cohort Building

Search for clinical concepts, build concept sets, and define cohorts.

In [ ]:
def search_concepts(term: str, domain: str | None = None, vocab: str | None = None, limit: int = 25) -> pd.DataFrame:
    """Search OMOP vocabulary for concepts."""
    sql = """
        SELECT concept_id, concept_name, domain_id, vocabulary_id,
               concept_class_id, standard_concept, concept_code
        FROM omop.concept
        WHERE concept_name ILIKE :term
          AND invalid_reason IS NULL
    """
    params: dict = {'term': f'%{term}%'}
    if domain:
        sql += ' AND domain_id = :domain'
        params['domain'] = domain
    if vocab:
        sql += ' AND vocabulary_id = :vocab'
        params['vocab'] = vocab
    sql += ' ORDER BY concept_name LIMIT :limit'
    params['limit'] = limit
    return query(sql, params)

def concept_hierarchy(concept_id: int) -> pd.DataFrame:
    """Get ancestors and descendants of a concept."""
    return query("""
        SELECT 'ancestor' AS direction, ca.ancestor_concept_id AS related_id,
               c.concept_name, c.domain_id, ca.min_levels_of_separation AS distance
        FROM omop.concept_ancestor ca
        JOIN omop.concept c ON ca.ancestor_concept_id = c.concept_id
        WHERE ca.descendant_concept_id = :cid AND ca.ancestor_concept_id != :cid
        UNION ALL
        SELECT 'descendant', ca.descendant_concept_id,
               c.concept_name, c.domain_id, ca.min_levels_of_separation
        FROM omop.concept_ancestor ca
        JOIN omop.concept c ON ca.descendant_concept_id = c.concept_id
        WHERE ca.ancestor_concept_id = :cid AND ca.descendant_concept_id != :cid
        ORDER BY direction, distance
        LIMIT 50
    """, {'cid': concept_id})

def concept_set_prevalence(concept_ids: list[int]) -> pd.DataFrame:
    """Check how many patients have records for a set of concepts."""
    placeholders = ','.join(str(int(cid)) for cid in concept_ids)
    return query(f"""
        SELECT c.concept_id, c.concept_name, c.domain_id,
               COUNT(DISTINCT co.person_id) AS patients
        FROM omop.concept c
        LEFT JOIN omop.condition_occurrence co ON c.concept_id = co.condition_concept_id
        WHERE c.concept_id IN ({placeholders})
        GROUP BY c.concept_id, c.concept_name, c.domain_id
        ORDER BY patients DESC
    """)

print('\u2713 Phenotype helpers ready')

In [ ]:
# Search for Type 2 Diabetes concepts
t2dm = search_concepts('type 2 diabetes', domain='Condition')
display(t2dm.head(10))

## 4. Clinical Domain — Observational Analytics

Run cohort operations, characterizations, and outcome analyses via the FinnGen runner (Darkstar R backend).

In [ ]:
def run_cohort_operations(source: dict, cohort_definition: dict, **kwargs) -> dict:
    """Submit a cohort operation to the FinnGen runner."""
    payload = {
        'source': source,
        'cohort_definition': cohort_definition,
        **kwargs
    }
    return api_post('/study-agent/finngen/cohort-operations', json=payload)

def run_co2_analysis(source: dict, module_key: str, **kwargs) -> dict:
    """Submit a CO2 (Clinical Outcome Observational) analysis."""
    payload = {
        'source': source,
        'module_key': module_key,
        **kwargs
    }
    return api_post('/study-agent/finngen/co2-analysis', json=payload)

def list_runs(investigation_id: int | None = None) -> pd.DataFrame:
    """List FinnGen analysis runs, optionally filtered by investigation."""
    sql = """
        SELECT id, service_name, status, investigation_id, created_at, updated_at
        FROM app.finngen_runs
    """
    params = {}
    if investigation_id:
        sql += ' WHERE investigation_id = :inv_id'
        params['inv_id'] = investigation_id
    sql += ' ORDER BY created_at DESC LIMIT 20'
    return query(sql, params)

# Default source configuration — adjust to your data source
DEFAULT_SOURCE = {
    'sourceKey': 'acumenus',
    'sourceName': 'Acumenus',
    'cdmSchema': 'omop',
    'resultsSchema': 'results',
    'vocabSchema': 'omop',
    'tempSchema': 'temp',
}

print('\u2713 Clinical analytics helpers ready')
print(f'   Default source: {DEFAULT_SOURCE["sourceName"]}')

In [ ]:
# Build a cohort from SQL — patients with T2DM and at least one visit
t2dm_cohort = query("""
    SELECT DISTINCT co.person_id,
           MIN(co.condition_start_date) AS index_date,
           p.gender_concept_id,
           EXTRACT(YEAR FROM MIN(co.condition_start_date)) - p.year_of_birth AS age_at_index
    FROM omop.condition_occurrence co
    JOIN omop.person p ON co.person_id = p.person_id
    WHERE co.condition_concept_id IN (
        SELECT descendant_concept_id
        FROM omop.concept_ancestor
        WHERE ancestor_concept_id = 201826  -- Type 2 diabetes mellitus
    )
    GROUP BY co.person_id, p.gender_concept_id, p.year_of_birth
""")
print(f'T2DM cohort: {len(t2dm_cohort)} patients')
t2dm_cohort.head()

## 5. Genomic Domain — GWAS, Open Targets, Colocalization

Query genomic evidence from Open Targets Platform and GWAS Catalog via the Parthenon API proxy.

In [ ]:
def search_open_targets(query_term: str, entity_type: str = 'disease') -> dict:
    """Search Open Targets Platform for diseases, targets, or drugs."""
    return api_get(f'/genomic-evidence/open-targets/search', params={
        'q': query_term,
        'entity': entity_type
    })

def get_disease_associations(disease_id: str, limit: int = 25) -> dict:
    """Get target associations for a disease from Open Targets."""
    return api_get(f'/genomic-evidence/open-targets/associations', params={
        'disease_id': disease_id,
        'limit': limit
    })

def search_gwas_catalog(trait: str) -> dict:
    """Search the GWAS Catalog for associations by disease trait."""
    return api_get(f'/genomic-evidence/gwas-catalog/search', params={
        'trait': trait
    })

def gwas_by_gene(gene_symbol: str) -> dict:
    """Search GWAS Catalog for associations by gene."""
    return api_get(f'/genomic-evidence/gwas-catalog/gene', params={
        'gene': gene_symbol
    })

print('\u2713 Genomic evidence helpers ready')

In [ ]:
# Search Open Targets for Type 2 Diabetes
try:
    ot_results = search_open_targets('type 2 diabetes', 'disease')
    hits = ot_results.get('data', {}).get('hits', [])
    if hits:
        ot_df = pd.DataFrame([{
            'id': h.get('id'),
            'name': h.get('name'),
            'score': h.get('score'),
        } for h in hits[:10]])
        display(ot_df)
    else:
        print('No results')
except Exception as e:
    print(f'Open Targets unavailable: {e}')

In [ ]:
# Search GWAS Catalog for diabetes associations
try:
    gwas = search_gwas_catalog('type 2 diabetes')
    associations = gwas.get('data', {}).get('associations', [])
    if associations:
        gwas_df = pd.DataFrame([{
            'trait': a.get('diseaseTrait', {}).get('trait', ''),
            'gene': a.get('gene', ''),
            'p_value': a.get('pvalue', ''),
            'risk_allele': a.get('riskAllele', ''),
            'study': a.get('study', ''),
        } for a in associations[:15]])
        display(gwas_df)
    else:
        print('No GWAS associations found')
except Exception as e:
    print(f'GWAS Catalog unavailable: {e}')

## 6. Evidence Synthesis — Cross-Domain Analysis

Combine clinical and genomic evidence to build an Evidence Dossier.

In [ ]:
def pin_evidence(inv_id: int, domain: str, evidence_type: str, title: str,
                 content: dict, concept_ids: list[int] | None = None,
                 gene_symbols: list[str] | None = None) -> dict:
    """Pin a finding to an investigation's evidence dossier."""
    payload = {
        'domain': domain,
        'evidence_type': evidence_type,
        'title': title,
        'content': content,
    }
    if concept_ids:
        payload['concept_ids'] = concept_ids
    if gene_symbols:
        payload['gene_symbols'] = gene_symbols
    return api_post(f'/investigations/{inv_id}/pins', json=payload)

def cross_links(inv_id: int) -> dict:
    """Get cross-domain links for an investigation (shared concepts/genes between pins)."""
    return api_get(f'/investigations/{inv_id}/cross-links')

print('\u2713 Synthesis helpers ready')

In [ ]:
# Example: Review all pins for an investigation
# INV_ID = 1  # set to your investigation ID
# pins = get_pins(INV_ID)
# display(pins)
#
# # Check cross-domain links
# links = cross_links(INV_ID)
# print(f'Cross-links: {links}')

## 7. Visualizations

Publication-quality dark-themed charts matching the Parthenon clinical UI.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

COLORS = {
    'bg': '#0E0E11',
    'panel': '#18181B',
    'border': '#27272A',
    'text': '#C5C0B8',
    'crimson': '#9B1B30',
    'gold': '#C9A227',
    'teal': '#2DD4BF',
    'muted': '#71717A',
}

DOMAIN_COLORS = {
    'phenotype': COLORS['teal'],
    'clinical': COLORS['gold'],
    'genomic': COLORS['crimson'],
    'synthesis': '#818CF8',  # indigo
}

def evidence_fig(width=12, height=6, title=''):
    """Create a Parthenon-themed figure."""
    fig, ax = plt.subplots(figsize=(width, height), facecolor=COLORS['bg'])
    ax.set_facecolor(COLORS['bg'])
    ax.tick_params(colors=COLORS['text'], labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(COLORS['border'])
    if title:
        ax.set_title(title, color=COLORS['text'], fontsize=14, fontweight='bold', pad=12)
    return fig, ax

print('\u2713 Visualization helpers ready')

In [ ]:
# Age distribution of T2DM cohort (from Section 4)
if 'age_at_index' in t2dm_cohort.columns and not t2dm_cohort.empty:
    fig, ax = evidence_fig(10, 5, 'Age at T2DM Index Date')
    ax.hist(t2dm_cohort['age_at_index'].dropna(), bins=30,
            color=DOMAIN_COLORS['phenotype'], alpha=0.8, edgecolor=COLORS['border'])
    ax.set_xlabel('Age', color=COLORS['muted'])
    ax.set_ylabel('Patients', color=COLORS['muted'])
    plt.tight_layout()
    plt.show()
else:
    print('Run the T2DM cohort cell in Section 4 first')

In [ ]:
# Top conditions in CDM — useful for identifying study populations
top_conditions = query("""
    SELECT c.concept_name AS condition, COUNT(DISTINCT co.person_id) AS patients
    FROM omop.condition_occurrence co
    JOIN omop.concept c ON co.condition_concept_id = c.concept_id
    WHERE c.standard_concept = 'S'
    GROUP BY c.concept_name
    ORDER BY patients DESC
    LIMIT 15
""")

if not top_conditions.empty:
    fig, ax = evidence_fig(12, 6, 'Top 15 Conditions by Patient Count')
    bars = ax.barh(top_conditions['condition'][::-1], top_conditions['patients'][::-1],
                   color=DOMAIN_COLORS['clinical'], edgecolor=COLORS['border'], height=0.6)
    ax.set_xlabel('Patients', color=COLORS['muted'])
    plt.tight_layout()
    plt.show()

## 8. Quick Recipe Reference

Copy these into new cells for common FinnGen workbench tasks.

In [ ]:
recipes = pd.DataFrame([
    {'domain': 'Investigation', 'task': 'List investigations', 'code': 'list_investigations()'},
    {'domain': 'Investigation', 'task': 'Create investigation', 'code': "create_investigation('My Study', 'Does X affect Y?')"},
    {'domain': 'Investigation', 'task': 'Get pins', 'code': 'get_pins(1)'},
    {'domain': 'Phenotype', 'task': 'Search concepts', 'code': "search_concepts('hypertension', domain='Condition')"},
    {'domain': 'Phenotype', 'task': 'Concept hierarchy', 'code': 'concept_hierarchy(201826)'},
    {'domain': 'Phenotype', 'task': 'Concept prevalence', 'code': 'concept_set_prevalence([201826, 443238])'},
    {'domain': 'Clinical', 'task': 'Build SQL cohort', 'code': "query('SELECT person_id FROM omop.condition_occurrence WHERE ...')"},
    {'domain': 'Clinical', 'task': 'List runs', 'code': 'list_runs(investigation_id=1)'},
    {'domain': 'Genomic', 'task': 'Open Targets search', 'code': "search_open_targets('type 2 diabetes')"},
    {'domain': 'Genomic', 'task': 'GWAS by trait', 'code': "search_gwas_catalog('type 2 diabetes')"},
    {'domain': 'Genomic', 'task': 'GWAS by gene', 'code': "gwas_by_gene('TCF7L2')"},
    {'domain': 'Synthesis', 'task': 'Pin evidence', 'code': "pin_evidence(1, 'genomic', 'gwas_hit', 'TCF7L2 association', {...})"},
    {'domain': 'Synthesis', 'task': 'Cross-links', 'code': 'cross_links(1)'},
    {'domain': 'API', 'task': 'Health check', 'code': "api_get('/health')"},
])
recipes

---

## Next Steps

1. **Create an investigation** — start with a research question
2. **Search phenotypes** — use `search_concepts()` to find your target/comparator populations
3. **Build cohorts** — SQL-based cohort construction with the OMOP CDM
4. **Query genomic evidence** — Open Targets and GWAS Catalog for genetic associations
5. **Pin findings** — accumulate evidence into your investigation's dossier
6. **Synthesize** — use `cross_links()` to find connections between clinical and genomic evidence
7. **Export** — save DataFrames to CSV/Parquet for publication or further analysis

**Evidence domains:** Phenotype (cohort definition) | Clinical (observational analytics) | Genomic (GWAS, Open Targets) | Synthesis (dossier assembly)

**FinnGen services:** Cohort Operations, CO2 Analysis, HADES Extras, ROMOPAPI — all accessible via the study-agent API.

Your server stops after 30 minutes of inactivity. All work is saved automatically.